# M3 – Encoding & Feature Selection (Regression)

**Project:** IT3051 FDM Mini Project 2026 – DataCo Smart Supply Chain  
**Owner:** M3 – Primesh (Encoding & Frontend)  
**Task:** Regression on **`Days for shipping (real)`**  
**Module:** `src/encoders.py`

| Step | What happens |
|------|-------------|
| 0 | Setup: imports, paths, settings, `src/encoders.py` |
| 1 | Load training data and grouped CV folds |
| 2 | Cardinality, rare categories and unseen categories |
| 3 | Target encoding: leakage demo |
| 4 | Compare four encoding strategies |
| 5 | Feature selection: correlation, near-constant, drop-column importance, backward selection |
| 6 | Save `results/m3_selection.json` and log experiments |
| 7 | Final check: fit preprocessor on train, transform test |
| 8 | Viva summary |

**Before running:** make sure `data/processed/train_reg.csv` and `test_reg.csv` exist (produced by M1's notebook).

## Step 0 – Setup

### 0.1 Imports, paths and settings

In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
PROCESSED_DIR = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "results"
FIG_DIR = ROOT / "reports" / "figures" / "m3"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "Days for shipping (real)"
GROUP = "Order Id"
SEED = 42
FAST = False          # True = use 30% of orders for a quick run while developing

def show(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{name}.png", dpi=120)
    plt.show()

### 0.2 The encoding module – `src/encoders.py`

The full encoding logic lives in `src/encoders.py` (not a notebook cell) so the trained pipeline can be pickled and loaded by the backend.

| Part | Purpose |
|------|---------|
| `LOW_CARD_MAX = 30` | Columns with ≤ 30 categories → one-hot; more → high-cardinality strategy |
| `ENGINEERED` | M2's feature groups and the raw columns each one needs |
| `FrequencyEncoder` | Replaces each category with its share of training rows; unseen → 0 |
| `high_card_encoder(strategy)` | Returns the encoder for a chosen strategy |
| `split_columns(X, features)` | Sorts raw columns into numeric / low-cardinality / high-cardinality |
| `build_preprocessor(X, features, high_card)` | Builds the full `ColumnTransformer` for any feature list and strategy |

**High-cardinality strategies:**

| Strategy | What it does | Pro | Con |
|----------|-------------|-----|-----|
| `drop` | Leaves the column out | Simplest | Loses any signal |
| `onehot_rare` | One-hot for ≥ 1% of rows; rest → `infrequent` | Keeps common categories exactly | Rare lumped together |
| `frequency` | Replace with training-set frequency | One number per column, no target used | Two categories with same frequency look identical |
| `target` | Replace with per-category average target | Directly captures each category's effect | Can leak if done wrongly (see Step 3) |

---

## Step 1 – Load the training data and the grouped CV folds

### 1.1 Data, folds and candidate features

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from src.features import DATE_COL
from src.encoders import ENGINEERED, LOW_CARD_MAX, FrequencyEncoder, build_preprocessor, split_columns

train = pd.read_csv(PROCESSED_DIR / "train_reg.csv")
if FAST:
    keep = pd.Series(train[GROUP].unique()).sample(frac=0.3, random_state=SEED)
    train = train[train[GROUP].isin(keep)].reset_index(drop=True)

X, y, groups = train.drop(columns=[TARGET]), train[TARGET], train[GROUP]

# Same kind of folds as M1: grouped by order, stratified on the target. Created once, reused everywhere.
cv_splits = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(X, y, groups))

ALL_FEATURES = [c for c in X.columns if c not in (GROUP, DATE_COL)] + [u for u in ENGINEERED if all(c in X.columns for c in ENGINEERED[u])]
print(f"Train: {len(X):,} rows, {groups.nunique():,} orders | {len(cv_splits)} grouped folds")
print("Candidate features:", ALL_FEATURES)

---

## Step 2 – Categorical columns: cardinality, rare and unseen categories

### 2.1 Cardinality table

In [ ]:
NUMERIC, LOW_CARD, HIGH_CARD = split_columns(X, ALL_FEATURES)
CATEGORICAL = LOW_CARD + HIGH_CARD

def rare_share(s, min_rows=30):
    counts = s.value_counts()
    return (counts < min_rows).mean() * 100

card = pd.DataFrame({
    "n_categories": [X[c].nunique() for c in CATEGORICAL],
    "top_category_%": [X[c].value_counts(normalize=True).iloc[0] * 100 for c in CATEGORICAL],
    "rare_categories_%": [rare_share(X[c]) for c in CATEGORICAL],
    "group": ["low (one-hot)" if c in LOW_CARD else "HIGH" for c in CATEGORICAL],
}, index=CATEGORICAL).sort_values("n_categories", ascending=False)
card.round(1)

### 2.2 Long tail of the biggest column

In [ ]:
col = card.index[0]                                    # the column with the most categories
coverage = X[col].value_counts(normalize=True).cumsum().reset_index(drop=True) * 100
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(np.arange(1, len(coverage) + 1), coverage)
ax.set_xlabel(f"Number of most common {col} values"); ax.set_ylabel("% of rows covered")
ax.set_title(f"Long tail of {col}")
show(fig, "01_long_tail")
for n in [10, 50, 100, 500]:
    if n <= len(coverage):
        print(f"Top {n:>4} values cover {coverage.iloc[n - 1]:.1f}% of rows")

### 2.3 Unseen categories in validation

In [ ]:
# How often does a validation fold contain a category the training folds never saw?
unseen = {}
for c in HIGH_CARD:
    shares = []
    for tr_idx, va_idx in cv_splits:
        seen = set(X[c].iloc[tr_idx])
        shares.append((~X[c].iloc[va_idx].isin(seen)).mean() * 100)
    unseen[c] = np.mean(shares)
pd.Series(unseen, name="unseen_in_validation_%").round(2)

---

## Step 3 – Target encoding and leakage

### 3.1 Naive vs pipeline target encoding

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import TargetEncoder

SCORING = {"MAE": "neg_mean_absolute_error", "RMSE": "neg_root_mean_squared_error", "R2": "r2"}
leak_col = card.index[0]

# WRONG: category means computed once on ALL training rows, then cross-validated
naive_feature = y.groupby(X[leak_col]).transform("mean").to_frame()
naive = cross_validate(LinearRegression(), naive_feature, y, cv=cv_splits, scoring=SCORING)

# RIGHT: TargetEncoder inside the pipeline -> learned from the training folds only
proper_pipe = Pipeline([("encode", TargetEncoder(target_type="continuous", random_state=SEED)),
                        ("model", LinearRegression())])
proper = cross_validate(proper_pipe, X[[leak_col]], y, cv=cv_splits, scoring=SCORING)

pd.DataFrame({
    "naive (leaky)": {"MAE": -naive["test_MAE"].mean(), "R2": naive["test_R2"].mean()},
    "inside pipeline": {"MAE": -proper["test_MAE"].mean(), "R2": proper["test_R2"].mean()},
}).round(4)

---

## Step 4 – Compare encoding strategies

### 4.1 Cross-validated comparison

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor

STRATEGIES = ["drop", "onehot_rare", "frequency", "target"]
MODELS = {
    "Ridge": lambda: Pipeline([("scale", StandardScaler()), ("model", Ridge(alpha=1.0))]),
    "HistGB": lambda: HistGradientBoostingRegressor(max_iter=200, random_state=SEED),
}

def evaluate(features, strategy, model_name):
    pipe = Pipeline([("prep", build_preprocessor(X, features, high_card=strategy)),
                     ("model", MODELS[model_name]())])
    s = cross_validate(pipe, X, y, cv=cv_splits, scoring=SCORING)
    return {"MAE": -s["test_MAE"].mean(), "RMSE": -s["test_RMSE"].mean(),
            "R2": s["test_R2"].mean(), "MAE_std": s["test_MAE"].std()}

rows = []
for strategy in STRATEGIES:
    for model_name in MODELS:
        rows.append({"strategy": strategy, "model": model_name, **evaluate(ALL_FEATURES, strategy, model_name)})
enc_results = pd.DataFrame(rows)
enc_results.round(4)

### 4.2 Comparison chart

In [ ]:
pivot = enc_results.pivot(index="strategy", columns="model", values="MAE").loc[STRATEGIES]
ax = pivot.plot.bar(figsize=(6, 3))
ax.set_ylabel("Cross-validated MAE (days)"); ax.set_title("Encoding strategy for high-cardinality columns")
ax.set_ylim(pivot.min().min() * 0.95, pivot.max().max() * 1.02)
show(ax.figure, "02_encoding_comparison")

### 4.3 Pick the best strategy

In [ ]:
best = enc_results.sort_values("MAE").iloc[0]
BEST_STRATEGY = best["strategy"]
print(f"Best: {BEST_STRATEGY} with {best['model']} (MAE {best['MAE']:.4f} ± {best['MAE_std']:.4f})")

---

## Step 5 – Feature selection

### 5.1 Correlation filter

In [ ]:
# 5.1 Correlation filter: from each numeric pair with |r| > 0.9, drop the one less related to the target
num_corr = X[NUMERIC].corr()
target_rel = X[NUMERIC].corrwith(y, method="spearman").abs()
corr_drop = {}
for i, a in enumerate(NUMERIC):
    for b in NUMERIC[i + 1:]:
        r = num_corr.loc[a, b]
        if abs(r) > 0.9 and a not in corr_drop and b not in corr_drop:
            weaker = a if target_rel[a] < target_rel[b] else b
            stronger = b if weaker == a else a
            corr_drop[weaker] = f"|r| = {abs(r):.2f} with '{stronger}'"
print("Dropped by correlation filter:", corr_drop or "none")

### 5.2 Near-constant columns

In [ ]:
# 5.2 Near-constant categorical columns: one category covers more than 99% of rows
near_constant = card.index[card["top_category_%"] > 99].tolist()
print("Near-constant columns:", near_constant or "none")

CANDIDATES = [f for f in ALL_FEATURES if f not in corr_drop and f not in near_constant]
print(f"{len(CANDIDATES)} candidate features after filters")

### 5.3 Drop-column importance

In [ ]:
# 5.3 Drop-column importance on one held-out grouped fold:
#     retrain without each feature and measure how much worse the validation MAE gets
from sklearn.metrics import mean_absolute_error

tr_idx, va_idx = cv_splits[0]

def fold_mae(features):
    pipe = Pipeline([("prep", build_preprocessor(X, features, high_card=BEST_STRATEGY)),
                     ("model", HistGradientBoostingRegressor(max_iter=200, random_state=SEED))])
    pipe.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    return mean_absolute_error(y.iloc[va_idx], pipe.predict(X.iloc[va_idx]))

base_mae = fold_mae(CANDIDATES)
importance = pd.Series(
    {f: fold_mae([g for g in CANDIDATES if g != f]) - base_mae for f in CANDIDATES},
    name="MAE_increase_when_removed",
).sort_values(ascending=False)
print(f"Validation MAE with all {len(CANDIDATES)} candidates: {base_mae:.4f}")
importance.round(4).to_frame()

### 5.4 Importance chart

In [ ]:
fig, ax = plt.subplots(figsize=(6, max(3, 0.3 * len(importance))))
ax.barh(importance.index[::-1], importance.values[::-1])
ax.axvline(0, color="grey", lw=0.8)
ax.set_xlabel("Increase in MAE when the feature is removed (days)"); ax.set_title("Drop-column importance")
show(fig, "03_drop_column_importance")

### 5.5 Backward selection

In [ ]:
# 5.5 Backward selection: keep the top-k features and check the cross-validated MAE
ranked = importance.index.tolist()
ks = sorted({len(ranked), 12, 8, 5, 3, 1} - {0}, reverse=True)
ks = [k for k in ks if k <= len(ranked)]

sel_rows = []
for k in ks:
    sel_rows.append({"k": k, "features": ranked[:k], **evaluate(ranked[:k], BEST_STRATEGY, "HistGB")})
selection = pd.DataFrame(sel_rows)
selection[["k", "MAE", "MAE_std", "R2"]].round(4)

### 5.6 Backward selection chart

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.errorbar(selection["k"], selection["MAE"], yerr=selection["MAE_std"], marker="o", capsize=3)
ax.invert_xaxis()
ax.set_xlabel("Number of features kept (most important first)"); ax.set_ylabel("CV MAE (days)")
ax.set_title("Does dropping weak features hurt?")
show(fig, "04_backward_selection")

### 5.7 Choose the final feature set

In [ ]:
# Smallest feature set whose MAE is within one standard deviation of the best (the "one-standard-error" rule)
best_row = selection.loc[selection["MAE"].idxmin()]
ok = selection[selection["MAE"] <= best_row["MAE"] + best_row["MAE_std"]]
chosen = ok.loc[ok["k"].idxmin()]
SELECTED = chosen["features"]
dropped = {f: "low drop-column importance" for f in CANDIDATES if f not in SELECTED}
dropped.update({f: r for f, r in corr_drop.items()})
dropped.update({f: "near-constant" for f in near_constant})
print(f"Selected {len(SELECTED)} features (MAE {chosen['MAE']:.4f} vs best {best_row['MAE']:.4f}):")
print(SELECTED)

---

## Step 6 – Save the decision and log the experiments

### 6.1 Save the selection and the experiment log

In [ ]:
decision = {
    "target": TARGET,
    "high_card_strategy": BEST_STRATEGY,
    "low_card_max": LOW_CARD_MAX,
    "selected_features": list(SELECTED),
    "dropped_features": dropped,
    "cv": "StratifiedGroupKFold(5) grouped by Order Id",
    "selected_cv_MAE": round(float(chosen["MAE"]), 4),
}
(RESULTS_DIR / "m3_selection.json").write_text(json.dumps(decision, indent=2, ensure_ascii=False), encoding="utf-8")

log = pd.concat([
    enc_results.assign(owner="M3", experiment="encoding_" + enc_results["strategy"], n_features=len(ALL_FEATURES)),
    selection.assign(owner="M3", model="HistGB", experiment="selection_top" + selection["k"].astype(str), n_features=selection["k"]),
])[["owner", "experiment", "model", "n_features", "MAE", "MAE_std", "RMSE", "R2"]]
exp_path = RESULTS_DIR / "experiments.csv"
log.to_csv(exp_path, mode="a", header=not exp_path.exists(), index=False)
print("Saved m3_selection.json and", len(log), "rows to experiments.csv")

---

## Step 7 – Final check on the test set (transform only)

### 7.1 Fit on train, transform test

In [ ]:
test = pd.read_csv(PROCESSED_DIR / "test_reg.csv")
final_prep = build_preprocessor(X, SELECTED, high_card=BEST_STRATEGY)
Xt_tr = final_prep.fit_transform(X, y)            # target encoder needs y when fitting
Xt_te = final_prep.transform(test.drop(columns=[TARGET]))
print("Train matrix:", Xt_tr.shape, "| Test matrix:", Xt_te.shape)
print("Missing values:", bool(np.isnan(Xt_tr).any() or np.isnan(Xt_te).any()))
print("Output columns:", list(final_prep.get_feature_names_out())[:15], "...")

---

## Step 8 – Viva summary

### 8.1 Key results in one place

In [ ]:
print(f'''
HIGH-CARDINALITY  {", ".join(f"{c} ({X[c].nunique():,})" for c in HIGH_CARD)}
LOW-CARDINALITY   {", ".join(LOW_CARD)}  -> one-hot
LEAKAGE DEMO      naive target encoding R2 {naive["test_R2"].mean():.3f} vs inside pipeline {proper["test_R2"].mean():.3f}
ENCODING          best = {BEST_STRATEGY}  (CV MAE {best["MAE"]:.4f}, model {best["model"]})
CORR FILTER       dropped {list(corr_drop) or "none"}
NEAR-CONSTANT     dropped {near_constant or "none"}
TOP 3 FEATURES    {", ".join(ranked[:3])}
SELECTED          {len(SELECTED)} of {len(ALL_FEATURES)} features, CV MAE {chosen["MAE"]:.4f}
''')